# Đánh giá E5 + Reranker — corpus 5000 SP

Pipeline 2 giai đoạn:
1. **Bi-encoder** `e5_base_finetuned_5000` → retrieve top-**n**
2. **Cross-encoder reranker** → xếp hạng lại → lấy top-**k**

**Metrics:** Precision@k, Recall@k, **F1@k**, MRR@k, NDCG@k

> **Đánh giá trên `ecommerce.csv`:** query = `title`, corpus = `searchable_text`.  
> Script dò **Recall@n** trước → chọn `selected_n` đạt `--target-recall` → **chỉ rerank tại n đó** (không rerank max n).

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm einops

## 2) Setup: pull code + mount Drive + kiểm tra model

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if COLAB_REPO.exists() and (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=False)
elif not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO
SCRIPTS = REPO_DIR / "embedding_project" / "scripts"
sys.path.insert(0, str(SCRIPTS))

# Model trên Drive (sửa path nếu khác)
from google.colab import drive
drive.mount("/content/drive")
EMB_MODEL = Path("/content/drive/MyDrive/models/e5_base_finetuned_5000")
RERANKER = Path("/content/drive/MyDrive/models/reranker")

EVAL_CSV = REPO_DIR / "embedding_project" / "data" / "ecommerce.csv"
OUTPUT_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval.json"

for label, p in [("embedding", EMB_MODEL), ("reranker", RERANKER), ("csv", EVAL_CSV)]:
    print("OK" if p.exists() else "MISSING", label, "->", p)

## 3) Chọn n theo Recall, rồi mới rerank

1. Encode E5 → tính **Recall@n** → chọn `selected_n`
2. Load reranker → chỉ rerank tại `selected_n`
3. Grid k (5/10/20)

Cell eval dùng `--skip-threshold` (nhanh). **Ngưỡng triển khai** → chạy cell **5) Ngưỡng tối ưu** riêng sau.

In [ ]:
import torch

MAX_QUERIES = 500
QUERY_COL = "title"
N_VALUES = [10, 20, 30, 50, 75, 100]
N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
TARGET_RECALL = 0.98
K_VALUES = [5, 10, 20]
EVAL_K = 10
RERANK_BATCH = 32

OUTPUT_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval_dense_n.json"

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(EVAL_CSV),
    "--query-col", QUERY_COL,
    "--output", str(OUTPUT_JSON),
    "--eval-k", str(EVAL_K),
    "--target-recall", str(TARGET_RECALL),
    "--n-values", *[str(n) for n in N_VALUES],
    "--n-search-values", *[str(n) for n in N_SEARCH_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
    "--rerank-batch-size", str(RERANK_BATCH),
    "--skip-threshold",
]
if MAX_QUERIES:
    cmd.extend(["--max-queries", str(MAX_QUERIES)])

print("CUDA:", torch.cuda.is_available())
print(f"Recall@n → chọn n nhỏ nhất đạt {TARGET_RECALL}, rồi mới rerank tại n đó.")
print("Lệnh:", " ".join(cmd))
!{" ".join(cmd)}

## 3b) Đánh giá query thực tế (`test_5000.jsonl`)

Query tự nhiên từ tập **test** (750 mẫu) — **không** dùng `title` làm query → metric thực tế hơn, bớt ảo.

Copy script mới trước khi chạy. Kết quả lưu file JSON riêng `*_test_jsonl.json`.

In [ ]:
import torch

TEST_JSONL = REPO_DIR / "data/training/test_5000.jsonl"
if not TEST_JSONL.is_file():
    raise FileNotFoundError(f"Thiếu {TEST_JSONL} — pull repo đầy đủ hoặc copy file test.")

MAX_QUERIES = None          # None = full 750 query test
TARGET_RECALL = 0.95        # test khó hơn → hạ target một chút
N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
K_VALUES = [5, 10, 20]
EVAL_K = 10
RERANK_BATCH = 32

OUTPUT_JSON_REAL = REPO_DIR / "embedding_project/outputs/evaluation/reranker_pipeline_eval_test_jsonl.json"

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(EVAL_CSV),
    "--query-jsonl", str(TEST_JSONL),
    "--output", str(OUTPUT_JSON_REAL),
    "--eval-k", str(EVAL_K),
    "--target-recall", str(TARGET_RECALL),
    "--n-values", *[str(n) for n in N_SEARCH_VALUES],
    "--n-search-values", *[str(n) for n in N_SEARCH_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
    "--rerank-batch-size", str(RERANK_BATCH),
    "--skip-threshold",
]
if MAX_QUERIES:
    cmd.extend(["--max-queries", str(MAX_QUERIES)])

print("CUDA:", torch.cuda.is_available())
print("Query thực tế từ:", TEST_JSONL)
print("Lệnh:", " ".join(cmd))
!{" ".join(cmd)}

In [ ]:
import json
import pandas as pd

EVAL_K = 10
RESULT_PATH = REPO_DIR / "embedding_project/outputs/evaluation/reranker_pipeline_eval_test_jsonl.json"

if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Chưa có {RESULT_PATH} — chạy cell 3b trước.")

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
best_metric = result.get("best_metric", "NDCG")
selected_n = result.get("selected_n")
opt_nk = result["optimal_n_k"]

bi = result[f"bi_encoder_only@{EVAL_K}"]
rk_info = result[f"reranker_best_n@{EVAL_K}"]
rk = rk_info["metrics"]

def row_metrics(stage: str, m: dict) -> dict:
    return {
        "stage": stage,
        f"P@{EVAL_K}": m.get(f"Precision@{EVAL_K}", 0.0),
        f"R@{EVAL_K}": m.get(f"Recall@{EVAL_K}", 0.0),
        f"F1@{EVAL_K}": m.get(f"F1@{EVAL_K}", 0.0),
        f"MRR@{EVAL_K}": m.get(f"MRR@{EVAL_K}", 0.0),
        f"NDCG@{EVAL_K}": m.get(f"NDCG@{EVAL_K}", 0.0),
    }

print(f"=== Query thực tế | {result.get('eval_source')} ===")
print(f"{result['n_eval_queries']} queries | corpus {result['corpus_size']} | selected_n={selected_n}")

summary = pd.DataFrame([
    row_metrics("Bi-encoder only", bi),
    row_metrics(f"Reranker (n={rk_info['n']})", rk),
])
display(summary)

grid = pd.DataFrame(result["grid_search_n_k"])
if len(grid):
    rows = []
    for _, r in grid.sort_values("k").iterrows():
        k = int(r["k"])
        rows.append({"k": k, "P": r[f"Precision@{k}"], "R": r[f"Recall@{k}"], "F1": r[f"F1@{k}"], "NDCG": r[f"NDCG@{k}"]})
    display(pd.DataFrame(rows))

print("Recall@n:", json.dumps(opt_nk.get("recall_by_n", {}), indent=2))

## 3c) 200 query trên corpus 5000 SP (`ecommerce.csv`)

| | File |
|---|---|
| **Corpus (5000 SP)** | `embedding_project/data/ecommerce.csv` → `searchable_text` |
| **200 query** | `embedding_project/data/benchmark_queries_200.csv` |

Nhãn yếu: token query khớp trong `searchable_text` (~170/200 query).  
Output: P/R/F1@10, n/k tối ưu, ngưỡng EER. Copy script mới trước khi chạy.

In [ ]:
import pandas as pd
import torch

BENCHMARK_CSV = REPO_DIR / "embedding_project/data/benchmark_queries_200.csv"
CORPUS_CSV = REPO_DIR / "embedding_project/data/ecommerce.csv"

for p, name in [(CORPUS_CSV, "corpus"), (BENCHMARK_CSV, "benchmark queries")]:
    if not p.is_file():
        raise FileNotFoundError(f"Thiếu {name}: {p}")

n_products = len(pd.read_csv(CORPUS_CSV))
n_queries = len(pd.read_csv(BENCHMARK_CSV))
print(f"Corpus: {CORPUS_CSV.name} → {n_products} SP")
print(f"Queries: {BENCHMARK_CSV.name} → {n_queries} query")

TARGET_RECALL = 0.90
N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
K_VALUES = [5, 10, 20]
EVAL_K = 10
RERANK_BATCH = 32

OUTPUT_BENCH = REPO_DIR / "embedding_project/outputs/evaluation/reranker_pipeline_eval_benchmark_200.json"

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(CORPUS_CSV),
    "--benchmark-queries", str(BENCHMARK_CSV),
    "--output", str(OUTPUT_BENCH),
    "--eval-k", str(EVAL_K),
    "--target-recall", str(TARGET_RECALL),
    "--n-values", *[str(n) for n in N_SEARCH_VALUES],
    "--n-search-values", *[str(n) for n in N_SEARCH_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
    "--rerank-batch-size", str(RERANK_BATCH),
]
print("CUDA:", torch.cuda.is_available())
print("→ 200 query tìm trong 5000 SP, rerank + ngưỡng")
print("Lệnh:", " ".join(cmd))
!{" ".join(cmd)}

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

EVAL_K = 10
RESULT_PATH = REPO_DIR / "embedding_project/outputs/evaluation/reranker_pipeline_eval_benchmark_200.json"

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
meta = result.get("eval_meta", {})
bi = result[f"bi_encoder_only@{EVAL_K}"]
rk_info = result[f"reranker_best_n@{EVAL_K}"]
rk = rk_info["metrics"]
opt_nk = result["optimal_n_k"]
thr = result.get("threshold_analysis", {})
deploy = result.get("deployment_threshold", {})

print("=== Benchmark 200 ===")
print("Source:", result.get("eval_source"))
print("Labeling:", meta.get("labeling"))
print("Skipped (no label):", meta.get("benchmark_skipped_queries"))
print(f"Queries eval: {result['n_eval_queries']} | selected_n: {result.get('selected_n')}")

display(pd.DataFrame([
    {"stage": "Bi-encoder", "P@10": bi[f"Precision@{EVAL_K}"], "R@10": bi[f"Recall@{EVAL_K}"],
     "F1@10": bi[f"F1@{EVAL_K}"], "MRR@10": bi[f"MRR@{EVAL_K}"], "NDCG@10": bi[f"NDCG@{EVAL_K}"]},
    {"stage": f"Reranker n={rk_info['n']}", "P@10": rk[f"Precision@{EVAL_K}"],
     "R@10": rk[f"Recall@{EVAL_K}"], "F1@10": rk[f"F1@{EVAL_K}"],
     "MRR@10": rk[f"MRR@{EVAL_K}"], "NDCG@10": rk[f"NDCG@{EVAL_K}"]},
]))

print("\nRecall@n:", json.dumps(opt_nk.get("recall_by_n", {}), indent=2))

rows = []
for r in result["grid_search_n_k"]:
    k = int(r["k"])
    rows.append({"k": k, "P": r[f"Precision@{k}"], "R": r[f"Recall@{k}"], "F1": r[f"F1@{k}"], "NDCG": r[f"NDCG@{k}"]})
print("\n=== Metric theo k (n=selected_n) ===")
display(pd.DataFrame(rows).sort_values("k"))

if deploy.get("eer"):
    eer, me = deploy["eer"], deploy["min_error_rate"]
    print(f"\n=== Ngưỡng (hard neg từ top-n E5) ===")
    display(pd.DataFrame([
        {"loại": "EER", "τ": eer["threshold"], "FPR": eer["FPR"], "FNR": eer["FNR"], "err": eer["error_rate"]},
        {"loại": "Min error", "τ": me["threshold"], "FPR": me["FPR"], "FNR": me["FNR"], "err": me["error_rate"]},
    ]))
    curve = pd.DataFrame(result.get("threshold_curve", []))
    if len(curve):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
        ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
        ax[0].axvline(eer["threshold"], ls="--", color="gray")
        ax[0].legend(); ax[0].set_title("FPR/FNR vs τ")
        ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
        ax[1].axvline(me["threshold"], ls="--", color="navy")
        ax[1].set_title("Error rate vs τ")
        plt.tight_layout(); plt.show()

## 4) Bảng kết quả & biểu đồ ngưỡng

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

EVAL_K = 10
RESULT_PATH = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval_dense_n.json"

if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Chưa có kết quả: {RESULT_PATH}\nChạy cell eval trước.")

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
best_metric = result.get("best_metric", "NDCG")
selected_n = result.get("selected_n")
opt_nk = result["optimal_n_k"]

bi = result[f"bi_encoder_only@{EVAL_K}"]
rk_info = result[f"reranker_best_n@{EVAL_K}"]
rk = rk_info["metrics"]
opt = opt_nk[f"by_max_{best_metric.lower()}_at_k{EVAL_K}"]
thr = result.get("threshold_analysis", {})

def compact_metrics(row: dict) -> dict:
    k = int(row["k"])
    return {
        "n": row["n"],
        "k": k,
        "P": row[f"Precision@{k}"],
        "R": row[f"Recall@{k}"],
        "F1": row[f"F1@{k}"],
        "MRR": row[f"MRR@{k}"],
        "NDCG": row[f"NDCG@{k}"],
    }

grid = pd.DataFrame([compact_metrics(r) for r in result["grid_search_n_k"]]).sort_values("k")

def row_metrics(stage: str, m: dict) -> dict:
    return {
        "stage": stage,
        f"P@{EVAL_K}": m.get(f"Precision@{EVAL_K}", 0.0),
        f"R@{EVAL_K}": m.get(f"Recall@{EVAL_K}", 0.0),
        f"F1@{EVAL_K}": m.get(f"F1@{EVAL_K}", 0.0),
        f"MRR@{EVAL_K}": m.get(f"MRR@{EVAL_K}", 0.0),
        f"NDCG@{EVAL_K}": m.get(f"NDCG@{EVAL_K}", 0.0),
    }

summary = pd.DataFrame([
    row_metrics("Bi-encoder only", bi),
    row_metrics(f"Reranker (n={rk_info['n']})", rk),
])
print(f"=== So sánh @{EVAL_K} ({result['n_eval_queries']} queries, corpus {result['corpus_size']}) ===")
display(summary)

print(f"\n=== Reranker tại n={selected_n} — metric theo k ===")
display(grid)

print(f"=== selected_n = {selected_n} | rerank {result.get('rerank_pairs_scored_once', '?')} cặp ===")
print("Recall@n:", json.dumps(opt_nk.get("recall_by_n", {}), indent=2))

best_any = opt_nk.get(f"by_max_{best_metric.lower()}_any_k")
print(f"\n=== k tối ưu (max {best_metric}, n={selected_n}) ===")
print(json.dumps(best_any, ensure_ascii=False, indent=2))

print(f"\n=== Báo cáo chính @{EVAL_K} (max {best_metric}@{EVAL_K}) ===")
print(json.dumps(opt, ensure_ascii=False, indent=2))

if thr.get("EER"):
    print("\n=== Ngưỡng reranker ===")
    print("EER:", json.dumps(thr["EER"], indent=2))
    curve = pd.DataFrame(result.get("threshold_curve", []))
    if len(curve):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
        ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
        ax[0].legend()
        ax[0].set_title("FPR / FNR vs threshold")
        ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
        ax[1].set_title("Error rate vs threshold")
        plt.tight_layout()
        plt.show()
else:
    print("\n(Ngưỡng — chạy cell «5) Ngưỡng tối ưu» bên dưới)")

pivot = grid.pivot(index="n", columns="k", values="F1")
print("\n=== Bảng F1 (n × k) ===")
display(pivot)

## 5) Ngưỡng tối ưu trước triển khai

**Trước khi chạy:** copy file `evaluate_reranker_pipeline.py` **mới** (có `run_threshold_only`) vào  
`/content/llm_provider_benchmarking/embedding_project/scripts/` — ghi đè file cũ.

Cell gọi Python trực tiếp (không cần `--threshold-only` trên CLI).

In [ ]:
import importlib
import json
import sys
from argparse import Namespace

import matplotlib.pyplot as plt
import pandas as pd
import torch

MAX_QUERIES = 500
QUERY_COL = "title"
RERANK_BATCH = 32
THRESHOLD_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_threshold.json"
SCRIPT_PATH = SCRIPTS / "evaluate_reranker_pipeline.py"

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"Thiếu script. Copy file vào:\n  {SCRIPT_PATH}")

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))
import evaluate_reranker_pipeline as erp
importlib.reload(erp)

if not hasattr(erp, "run_threshold_only"):
    raise RuntimeError(
        "Script trên Colab là bản CŨ (thiếu run_threshold_only).\n"
        f"Copy file mới từ máy local ghi đè:\n  {SCRIPT_PATH}"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
args = Namespace(
    reranker_model=RERANKER,
    eval_csv=EVAL_CSV,
    query_col=QUERY_COL,
    output=OUTPUT_JSON,
    threshold_output=THRESHOLD_JSON,
    rerank_batch_size=RERANK_BATCH,
    max_queries=MAX_QUERIES,
    max_neg_per_query=3,
    query_jsonl=None,
    device=device,
)
print(f"Device: {device} | queries: {MAX_QUERIES or 'full'}")
erp.run_threshold_only(args, device)

if not THRESHOLD_JSON.is_file():
    raise FileNotFoundError(f"Chạy threshold thất bại — không có {THRESHOLD_JSON}")

thr_data = json.loads(THRESHOLD_JSON.read_text(encoding="utf-8"))
ta = thr_data["threshold_analysis"]
deploy = thr_data["deployment_threshold"]
eer = deploy["eer"]
min_err = deploy["min_error_rate"]

print(f"\n=== Dataset ngưỡng: {ta['n_pairs']} cặp ({ta['n_positive']} pos, {ta['n_negative']} neg) ===")
display(pd.DataFrame([
    {"loại": "EER (FPR ≈ FNR)", "τ": eer["threshold"], "FPR": eer["FPR"], "FNR": eer["FNR"], "error_rate": eer["error_rate"]},
    {"loại": "Min error rate", "τ": min_err["threshold"], "FPR": min_err["FPR"], "FNR": min_err["FNR"], "error_rate": min_err["error_rate"]},
]))

print(f"\n→ Triển khai: chỉ giữ SP có score reranker >= τ")
print(f"   EER (cân bằng FP/FN): τ = {eer['threshold']:.4f}")
print(f"   Min error:            τ = {min_err['threshold']:.4f}")

curve = pd.DataFrame(thr_data["threshold_curve"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
ax[0].axvline(eer["threshold"], ls="--", color="gray", label=f"EER τ={eer['threshold']:.3f}")
ax[0].legend(); ax[0].set_title("FPR / FNR vs threshold")
ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
ax[1].axvline(min_err["threshold"], ls="--", color="navy", label=f"min err τ={min_err['threshold']:.3f}")
ax[1].legend(); ax[1].set_title("Error rate vs threshold")
plt.tight_layout(); plt.show()

## 6) Giải thích cho báo cáo GVHD

- **n**: chọn **n nhỏ nhất** đạt Recall ≥ target (0.98)
- **k**: số kết quả cuối sau rerank (grid 5/10/20)
- **F1@10** = 2·P·R/(P+R)
- **EER**: τ sao cho FPR(τ) ≈ FNR(τ)
- **Min error**: τ làm (FP+FN) nhỏ nhất
- **Triển khai**: lọc `score_reranker >= τ` trước khi trả kết quả